# 06 — Évaluation Complète DeepEval
Évaluation systématique avec les 5 métriques DeepEval sur la meilleure configuration identifiée.

In [ ]:
import sys, os, pandas as pd, time
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../src"))

from config import load_config
from retriever import retrieve_documents
from llm_chain import generate_answer
from evaluation_judge import create_judge
from experiments.runner import run_full_pipeline, run_deepeval_metrics
from deepeval.test_case import LLMTestCase

from notebooks.lib.reporter import load_benchmark, to_csv, to_excel, to_json
from notebooks.lib.plotter import histogram, boxplot, heatmap
from experiments.registry import ExperimentLog

In [ ]:
benchmark = load_benchmark()
cfg = load_config()
judge = create_judge(cfg.evaluation)

METRICS = ["faithfulness", "answer_relevancy",
           "contextual_precision", "contextual_recall"]

log = ExperimentLog(name="deepeval_full")
log.set_params(embedding=cfg.embedding.model, llm=cfg.llm.model,
              judge=f"{cfg.evaluation.provider}:{cfg.evaluation.model}",
              top_k=cfg.retrieval.top_k, metrics=METRICS)

In [ ]:
for idx, row in benchmark.iterrows():
    q_id = row['ID']
    question = row['Question']
    expected = row['Ground_Truth']
    print(f"[{q_id}] {question[:60]}...")

    pipeline = run_full_pipeline(question, cfg.retrieval, cfg.llm)

    contexts = [d.page_content for d in pipeline['documents']]
    tc = LLMTestCase(
        input=question, actual_output=pipeline['answer'],
        expected_output=expected, retrieval_context=contexts,
    )

    metrics = run_deepeval_metrics(tc, judge, METRICS, cfg.evaluation.threshold)

    log.record(
        ID=q_id, niveau=row['Niveau_Complexite'],
        categorie=row['Categorie_Metier'],
        question=question,
        num_chunks=pipeline['num_chunks'],
        total_time_s=pipeline['total_time_s'],
        **metrics,
    )

paths = log.save_all()
log.append_to_global_log()
print(f"\nRésultats sauvegardés :")
for fmt, p in paths.items():
    print(f"  {fmt}: {p}")

In [ ]:
df = pd.DataFrame(log.results)

score_cols = [c for c in df.columns if c in
    ['faithfulness','answer_relevancy','contextual_precision','contextual_recall']]

print("=== Scores moyens ===")
print(df[score_cols].describe().round(4))

print("\n=== Taux de réussite (score >= 0.75) ===")
for col in score_cols:
    rate = (df[col] >= 0.75).mean() * 100
    print(f"  {col}: {rate:.1f}%")

In [ ]:
# Analyse par catégorie
print("=== Scores par Catégorie Métier ===")
cat = df.groupby('categorie')[score_cols].mean().round(4)
print(cat)

boxplot(df, x='categorie', y='faithfulness',
        title="Faithfulness par catégorie métier",
        filename='deepeval_faithfulness_by_category.png')

heatmap(cat, title="Scores moyens par catégorie",
        filename='deepeval_heatmap_category.png')

print("\nGraphiques sauvegardés dans outputs/figures/")